<a href="https://colab.research.google.com/github/FatemehNMT/Visual-SLAM-Book-Google-Colab/blob/main/Chapter_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In lecture 3, we saw that Eigen provided geometry modules but did not support Lie algebra. A better Lie algebra library is the Sophus library maintained by Strasdat (https://github.com/strasdat/Sophus).

The **Sophus** library supports SO(3) and SE(3), which are mainly discussed in this
chapter. In addition, it also contains two-dimensional motion SO(2), SE(2) and the similar transformation of Sim(3). It is developed directly on top of Eigen, and there is no need to install additional dependencies.

**Useful links:**

https://github.com/strasdat/Sophus

https://github.com/stonier/sophus

https://silenceoverflow.github.io/Awesome-SLAM/Misc/2018-03-10-Installing-Dependencies-on-Ubuntu-towards-SLAM-Projects.html

# **Chapter 3: Lie Group and Lie Algebra**

**Chapter Reference:** https://github.com/gaoxiang12/slambook2/blob/master/ch4

## **Mount**

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!pwd

/content


## **Install Eigen**

In [ ]:
!git clone https://gitlab.com/libeigen/eigen.git

Cloning into 'eigen'...
remote: Enumerating objects: 125758, done.
remote: Counting objects: 100% (462/462), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 125758 (delta 296), reused 448 (delta 288), pack-reused 125296 (from 1)
Receiving objects: 100% (125758/125758), 105.57 MiB | 19.17 MiB/s, done.
Resolving deltas: 100% (104172/104172), done.


In [ ]:
%cd eigen

/content/eigen


In [ ]:
!mkdir build
%cd build

/content/eigen/build


In [ ]:
!cmake ..

In [ ]:
!sudo make install

In [ ]:
!pwd

/content/eigen/build


In [ ]:
%cd /content/

/content


In [ ]:
# !rm -r Sophus

## **Install Sophus**

In [ ]:
!git clone https://github.com/strasdat/Sophus.git

Cloning into 'Sophus'...
remote: Enumerating objects: 23331, done.
remote: Counting objects: 100% (908/908), done.
remote: Compressing objects: 100% (360/360), done.
remote: Total 23331 (delta 514), reused 746 (delta 476), pack-reused 22423 (from 1)
Receiving objects: 100% (23331/23331), 261.54 MiB | 29.75 MiB/s, done.
Resolving deltas: 100% (15946/15946), done.


In [ ]:
%cd Sophus/

/content/Sophus


In [ ]:
!git clone https://github.com/Microsoft/vcpkg.git
%cd vcpkg

Cloning into 'vcpkg'...
remote: Enumerating objects: 251701, done.
remote: Counting objects: 100% (64641/64641), done.
remote: Compressing objects: 100% (6344/6344), done.
remote: Total 251701 (delta 60081), reused 58389 (delta 58296), pack-reused 187060 (from 1)
Receiving objects: 100% (251701/251701), 76.23 MiB | 21.90 MiB/s, done.
Resolving deltas: 100% (167815/167815), done.
/content/Sophus/vcpkg


In [ ]:
!./bootstrap-vcpkg.sh


In [ ]:
!./vcpkg integrate install

Applied user-wide integration for this vcpkg root.
CMake projects should use: "-DCMAKE_TOOLCHAIN_FILE=/content/Sophus/vcpkg/scripts/buildsystems/vcpkg.cmake"


In [ ]:
!./vcpkg install sophus

In [ ]:
!pwd

/content/Sophus/vcpkg


In [ ]:
%cd ../

/content/Sophus


In [ ]:
!mkdir build
%cd build

/content/Sophus/build


In [ ]:
!cmake ..



In [ ]:
!make

In [ ]:
!pwd

/content/Sophus/build


In [ ]:
%cd /content/

/content


In [ ]:
# go to the content folder
!mkdir MyExample
%cd MyExample

/content/MyExample


## **Examples**

### **SLAM Book 2: Basic Usage of Sophus**

**Page 71**\
https://github.com/gaoxiang12/slambook2/blob/master/ch4/useSophus.cpp

Let's demonstrate the SO(3) and SE(3) operations in the Sophus library:\
 \
The first half introduces the operation on SO(3), and the second half is SE(3).\
We demonstrate how to construct SO(3), SE(3) objects as well as the exponential/logarithm mapping. And then, we update the lie group elements when we know the updated amount.

In [ ]:
%%writefile useSophus.cpp

#include <iostream>
#include <cmath>
#include <Eigen/Core>
#include <Eigen/Geometry>
#include "sophus/se3.hpp"

using namespace std;
using namespace Eigen;

/// This program demonstrates the basic usage of sophus
int main(int argc, char **argv) {
  // Rotation matrix for 90 degrees along the Z axis
  Matrix3d R = AngleAxisd(M_PI / 2, Vector3d(0, 0, 1)).toRotationMatrix();
  // Or quaternion
  Quaterniond q(R);
  Sophus::SO3d SO3_R(R);              // Sophus::SO3d can be constructed directly from the rotation matrix.
  Sophus::SO3d SO3_q(q);              // It can also be constructed through quaternions.

  // The two are equivalent
  cout << "SO(3) from matrix:\n" << SO3_R.matrix() << endl;
  cout << "SO(3) from quaternion:\n" << SO3_q.matrix() << endl;
  cout << "they are equal" << endl;

  // Use the logarithmic map to obtain its Lie algebra
  Vector3d so3 = SO3_R.log();
  cout << "so3 = " << so3.transpose() << endl;

  // hat Vector to antisymmetric matrix
  cout << "so3 hat=\n" << Sophus::SO3d::hat(so3) << endl;

  // In contrast, vee is antisymmetric to the vector
  cout << "so3 hat vee= " << Sophus::SO3d::vee(Sophus::SO3d::hat(so3)).transpose() << endl;

  // Update of incremental perturbation model
  Vector3d update_so3(1e-4, 0, 0); //Assume the update amount is this much
  Sophus::SO3d SO3_updated = Sophus::SO3d::exp(update_so3) * SO3_R;
  cout << "SO3 updated = \n" << SO3_updated.matrix() << endl;

  cout << "*******************************" << endl;

  // The operation for SE(3) is similar
  Vector3d t(1, 0, 0);           // Translate along the X axis by 1
  Sophus::SE3d SE3_Rt(R, t);           // Construct SE(3) from R,t
  Sophus::SE3d SE3_qt(q, t);            // Construct SE(3) from q,t
  cout << "SE3 from R,t= \n" << SE3_Rt.matrix() << endl;
  cout << "SE3 from q,t= \n" << SE3_qt.matrix() << endl;

  // The Lie algebra se(3) is a six-dimensional vector. For convenience, we will typedef it as
  typedef Eigen::Matrix<double, 6, 1> Vector6d;
  Vector6d se3 = SE3_Rt.log();
  cout << "se3 = " << se3.transpose() << endl;

  // Observing the output, we can see that in Sophus, the translation of se(3) comes first and the rotation comes later.
  // Similarly, there are two operators: hat and vee
  cout << "se3 hat = \n" << Sophus::SE3d::hat(se3) << endl;
  cout << "se3 hat vee = " << Sophus::SE3d::vee(Sophus::SE3d::hat(se3)).transpose() << endl;

  // Finally, let’s demonstrate the update
  Vector6d update_se3; // Update volume
  update_se3.setZero();
  update_se3(0, 0) = 1e-4;
  Sophus::SE3d SE3_updated = Sophus::SE3d::exp(update_se3) * SE3_Rt;
  cout << "SE3 updated = " << endl << SE3_updated.matrix() << endl;

  return 0;
}

Writing useSophus.cpp


In [ ]:
%%writefile CMakeLists.txt

cmake_minimum_required(VERSION 3.0)
project(useSophus)

# To use sophu, you need to find it using the find_package command
# find_package(Sophus REQUIRED)
# link_directories(${Sophus_INCLUDE_DIRS})

include_directories("/content/Sophus")

# Eigen
include_directories("/usr/include/eigen3")

# ****************************************
# include_directories( "/usr/include/eigen3" )

# or

# find_package(Eigen3 REQUIRED)
# link_directories(${Eigen_INCLUDE_DIRS})

# Joftesham kar nakard .........????????????????????????

# or
include_directories("/content/eigen")
# ****************************************

add_executable(useSophus useSophus.cpp)
# target_link_libraries(useSophus Sophus::Sophus)

# add_subdirectory(example)

Overwriting CMakeLists.txt


In [ ]:
!pwd

/content/MyExample


#### **Using CMake**

In [ ]:
!mkdir build
%cd build/

/content/MyExample/build


In [ ]:
!cmake ..

CMake Deprecation Warning at CMakeLists.txt:2 (cmake_minimum_required):
  Compatibility with CMake < 3.5 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value or use a ...<max> suffix to tell
  CMake that the project does not need compatibility with older versions.


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Configuring done (0.4s)
-- Generating done (0.0s)
-- Build files have been written to: /content/MyExample/build


In [ ]:
!make useSophus

[ 50%] Building CXX object CMakeFiles/useSophus.dir/useSophus.cpp.o
[100%] Linking CXX executable useSophus
[100%] Built target useSophus


In [ ]:
!./useSophus

SO(3) from matrix:
2.22045e-16          -1           0
          1 2.22045e-16           0
          0           0           1
SO(3) from quaternion:
2.22045e-16          -1           0
          1 2.22045e-16           0
          0           0           1
they are equal
so3 =      0      0 1.5708
so3 hat=
      0 -1.5708       0
 1.5708       0      -0
     -0       0       0
so3 hat vee=      0      0 1.5708
SO3 updated = 
          0          -1           0
          1           0     -0.0001
     0.0001 2.03288e-20           1
*******************************
SE3 from R,t= 
2.22045e-16          -1           0           1
          1 2.22045e-16           0           0
          0           0           1           0
          0           0           0           1
SE3 from q,t= 
2.22045e-16          -1           0           1
          1 2.22045e-16           0           0
          0           0           1           0
          0           0           0           1
se3 =  0.785398 

#### **Without CMake**

In [ ]:
%cd ..

/content/MyExample


In [ ]:
!ls

build  CMakeLists.txt  useSophus.cpp


In [ ]:
%%script bash
# g++ useSophus.cpp -std=c++17 -o out
# g++ main.cpp -std=c++14 -o outt  -I /content/eigen
g++ useSophus.cpp -std=c++17 -o outt  -I /content/eigen -I /content/Sophus

In [ ]:
!./outt

SO(3) from matrix:
2.22045e-16          -1           0
          1 2.22045e-16           0
          0           0           1
SO(3) from quaternion:
2.22045e-16          -1           0
          1 2.22045e-16           0
          0           0           1
they are equal
so3 =      0      0 1.5708
so3 hat=
      0 -1.5708       0
 1.5708       0      -0
     -0       0       0
so3 hat vee=      0      0 1.5708
SO3 updated = 
          0          -1           0
          1           0     -0.0001
     0.0001 2.03288e-20           1
*******************************
SE3 from R,t= 
2.22045e-16          -1           0           1
          1 2.22045e-16           0           0
          0           0           1           0
          0           0           0           1
SE3 from q,t= 
2.22045e-16          -1           0           1
          1 2.22045e-16           0           0
          0           0           1           0
          0           0           0           1
se3 =  0.785398 

### **SLAM Book 2: Evaluating the Trajectory**

**Page 73**\
https://github.com/gaoxiang12/slambook2/blob/master/ch4/example/trajectoryError.cpp

**Evaluating the Trajectory**

In [ ]:
!pwd

/content/MyExample/build


In [ ]:
%cd ../../

/content


In [ ]:
!mkdir MyExample_2
%cd MyExample_2

/content/MyExample_2


In [ ]:
%%writefile CMakeLists.txt

option(USE_UBUNTU_20 "Set to ON if you are using Ubuntu 20.04" OFF)
# find_package(Pangolin REQUIRED)
if(USE_UBUNTU_20)
    message("You are using Ubuntu 20.04, fmt::fmt will be linked")
    find_package(fmt REQUIRED)
    set(FMT_LIBRARIES fmt::fmt)
endif()
# include_directories(${Pangolin_INCLUDE_DIRS})
add_executable(trajectoryError trajectoryError.cpp)
# target_link_libraries(trajectoryError ${Pangolin_LIBRARIES} ${FMT_LIBRARIES})

include_directories("/content/Sophus")

# Eigen
include_directories("/content/eigen")

Overwriting CMakeLists.txt


In [ ]:
%%writefile trajectoryError.cpp

#include <iostream>
#include <fstream>
#include <unistd.h>
// #include <pangolin/pangolin.h>
#include <sophus/se3.hpp>

using namespace Sophus;
using namespace std;

string groundtruth_file = "/content/MyExample_2/groundtruth.txt";
string estimated_file = "/content/MyExample_2/estimated.txt";
// /content/MyExample_2/estimated.txt

typedef vector<Sophus::SE3d, Eigen::aligned_allocator<Sophus::SE3d>> TrajectoryType;

void DrawTrajectory(const TrajectoryType &gt, const TrajectoryType &esti);

TrajectoryType ReadTrajectory(const string &path);

int main(int argc, char **argv) {
  TrajectoryType groundtruth = ReadTrajectory(groundtruth_file);
  TrajectoryType estimated = ReadTrajectory(estimated_file);
  assert(!groundtruth.empty() && !estimated.empty());
  assert(groundtruth.size() == estimated.size());

  // compute rmse
  double rmse = 0;
  for (size_t i = 0; i < estimated.size(); i++) {
    Sophus::SE3d p1 = estimated[i], p2 = groundtruth[i];
    double error = (p2.inverse() * p1).log().norm();
    rmse += error * error;
  }
  rmse = rmse / double(estimated.size());
  rmse = sqrt(rmse);
  cout << "RMSE = " << rmse << endl;

  // DrawTrajectory(groundtruth, estimated);
  return 0;
}

TrajectoryType ReadTrajectory(const string &path) {
  ifstream fin(path);
  TrajectoryType trajectory;
  if (!fin) {
    cerr << "trajectory " << path << " not found." << endl;
    return trajectory;
  }

  while (!fin.eof()) {
    double time, tx, ty, tz, qx, qy, qz, qw;
    fin >> time >> tx >> ty >> tz >> qx >> qy >> qz >> qw;
    Sophus::SE3d p1(Eigen::Quaterniond(qw, qx, qy, qz), Eigen::Vector3d(tx, ty, tz));
    trajectory.push_back(p1);
  }
  return trajectory;
}

// void DrawTrajectory(const TrajectoryType &gt, const TrajectoryType &esti) {
//   // create pangolin window and plot the trajectory
//   pangolin::CreateWindowAndBind("Trajectory Viewer", 1024, 768);
//   glEnable(GL_DEPTH_TEST);
//   glEnable(GL_BLEND);
//   glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA);

//   pangolin::OpenGlRenderState s_cam(
//       pangolin::ProjectionMatrix(1024, 768, 500, 500, 512, 389, 0.1, 1000),
//       pangolin::ModelViewLookAt(0, -0.1, -1.8, 0, 0, 0, 0.0, -1.0, 0.0)
//   );

//   pangolin::View &d_cam = pangolin::CreateDisplay()
//       .SetBounds(0.0, 1.0, pangolin::Attach::Pix(175), 1.0, -1024.0f / 768.0f)
//       .SetHandler(new pangolin::Handler3D(s_cam));


//   while (pangolin::ShouldQuit() == false) {
//     glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT);

//     d_cam.Activate(s_cam);
//     glClearColor(1.0f, 1.0f, 1.0f, 1.0f);

//     glLineWidth(2);
//     for (size_t i = 0; i < gt.size() - 1; i++) {
//       glColor3f(0.0f, 0.0f, 1.0f);  // blue for ground truth
//       glBegin(GL_LINES);
//       auto p1 = gt[i], p2 = gt[i + 1];
//       glVertex3d(p1.translation()[0], p1.translation()[1], p1.translation()[2]);
//       glVertex3d(p2.translation()[0], p2.translation()[1], p2.translation()[2]);
//       glEnd();
//     }

//     for (size_t i = 0; i < esti.size() - 1; i++) {
//       glColor3f(1.0f, 0.0f, 0.0f);  // red for estimated
//       glBegin(GL_LINES);
//       auto p1 = esti[i], p2 = esti[i + 1];
//       glVertex3d(p1.translation()[0], p1.translation()[1], p1.translation()[2]);
//       glVertex3d(p2.translation()[0], p2.translation()[1], p2.translation()[2]);
//       glEnd();
//     }
//     pangolin::FinishFrame();
//     usleep(5000);   // sleep 5 ms
//   }

// }

Overwriting trajectoryError.cpp


#### texts

In [ ]:
%%writefile estimated.txt

// 1305031526.67147303 ...
// Make a copy of this file in the current folder.


Writing estimated.txt


In [ ]:
%%writefile groundtruth.txt

// 1305031526.67210007 -0.0355094123046875154 ...
// Make a copy of this file in the current folder.


Writing groundtruth.txt


#### **Using CMake**

In [ ]:
!mkdir build
%cd build/

/content/MyExample_2/build


In [ ]:
!cmake ..

In [ ]:
!make trajectoryError

[ 50%] Building CXX object CMakeFiles/trajectoryError.dir/trajectoryError.o
[100%] Linking CXX executable trajectoryError
[100%] Built target trajectoryError


In [ ]:
!./trajectoryError

RMSE = 2.20727
